# 01 - Metrics Anomaly Detection

Notebook nay tap trung vao metrics de tra loi `WHEN` va mot phan `WHERE`.

Cong nghe/method da chot trong bai nop:
- `CUSUM` la detector chinh cho continuous drift detection
- `Isolation Forest` la detector phu cho multivariate anomaly
- `Rolling Z-score` duoc giu lai lam baseline de giai thich
- `STL` khong dung lam primary alert trong production, chi de giai thich hau kiem


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

BASE = Path(r"D:\AWS\AIOPS-study\g2-data\g2\metrics")
ART = Path(r"D:\AWS\AIOPS\w1\lab\artifacts")
ART.mkdir(parents=True, exist_ok=True)

ALERT_TIME = pd.Timestamp('2026-06-01T23:04:00Z')
EARLIEST_LOG_SIGNAL = pd.Timestamp('2026-06-01T06:30:19Z')
MEMORY_TREND_SHIFT = pd.Timestamp('2026-06-01T09:01:00Z')
SUSTAINED_MEMORY_TIME = pd.Timestamp('2026-06-01T16:20:30Z')
FIRST_OOMKILL = pd.Timestamp('2026-06-01T19:59:02Z')
FEATURES = ['memory_usage_bytes', 'jvm_gc_pause_ms_avg', 'http_p99_latency_ms', 'http_5xx_rate']

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fbfcfe',
    'axes.edgecolor': '#d0d7de',
    'grid.color': '#d8dee4',
    'grid.alpha': 0.35,
    'axes.titleweight': 'bold',
    'axes.titlesize': 14,
    'axes.labelsize': 11,
    'legend.frameon': True,
    'legend.framealpha': 0.95,
})


def fmt_time_axis(ax):
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H'))
    ax.tick_params(axis='x', rotation=0)


def add_reference_lines(ax, include_trend=True):
    if include_trend:
        ax.axvspan(MEMORY_TREND_SHIFT, ALERT_TIME, color='#f4a261', alpha=0.12, label='Silent window')
        ax.axvline(MEMORY_TREND_SHIFT, color='#f4a261', linestyle='--', linewidth=1.4, label='Memory trend shift')
    ax.axvline(ALERT_TIME, color='#c1121f', linestyle='--', linewidth=1.5, label='Alert fired')


In [ ]:
cart = pd.read_csv(BASE / 'cart-service.csv', parse_dates=['timestamp']).ffill().fillna(0)
api = pd.read_csv(BASE / 'api-gateway.csv', parse_dates=['timestamp']).ffill().fillna(0)
order = pd.read_csv(BASE / 'order-service.csv', parse_dates=['timestamp']).ffill().fillna(0)
payment = pd.read_csv(BASE / 'payment-service.csv', parse_dates=['timestamp']).ffill().fillna(0)
product = pd.read_csv(BASE / 'product-service.csv', parse_dates=['timestamp']).ffill().fillna(0)

cart['memory_utilization_pct'] = cart['memory_usage_bytes'] / cart['memory_limit_bytes'] * 100
restart_events = cart.loc[cart['container_restart_count'].diff().fillna(0) > 0, ['timestamp', 'container_restart_count']]

eda = pd.DataFrame([{
    'rows': len(cart),
    'first_memory_gt_60pct': cart.loc[cart['memory_utilization_pct'] > 60, 'timestamp'].min(),
    'first_cart_5xx_gt_5pct': cart.loc[cart['http_5xx_rate'] > 5, 'timestamp'].min(),
    'first_gateway_cart_error_gt_5pct': api.loc[api['cart_upstream_error_rate'] > 5, 'timestamp'].min(),
    'first_order_timeout_gt_5pct': order.loc[order['upstream_timeout_rate'] > 5, 'timestamp'].min(),
    'first_payment_timeout_gt_5pct': payment.loc[payment['upstream_timeout_rate'] > 5, 'timestamp'].min(),
    'restart_count_final': int(cart['container_restart_count'].max()),
}])
display(eda)
display(restart_events)


In [ ]:
def first_alarm_time(flags: pd.Series, timestamps: pd.Series, points_required: int = 3, window_points: int = 10):
    alarm = flags.astype(int).rolling(window_points, min_periods=window_points).sum() >= points_required
    matches = timestamps.loc[alarm]
    return matches.min() if not matches.empty else pd.NaT


def minutes_before_alert(ts):
    return None if pd.isna(ts) else int((ALERT_TIME - pd.Timestamp(ts)).total_seconds() // 60)


def duration_text(minutes):
    if minutes is None:
        return None
    h, m = divmod(minutes, 60)
    return f'{h}h {m:02d}m'


def run_cusum(series: pd.Series, baseline_points: int = 720, k: float = 0.5, h: float = 5.0, smoothing_points: int = 1):
    smooth = series.rolling(smoothing_points, min_periods=smoothing_points).mean()
    baseline = smooth.iloc[:baseline_points]
    mu = baseline.mean()
    sigma = baseline.std() or 1.0
    z = (smooth - mu) / sigma
    s_pos = []
    running = 0.0
    for value in z.fillna(0.0):
        running = max(0.0, running + value - k)
        s_pos.append(running)
    return pd.Series(s_pos, index=series.index), pd.Series(z, index=series.index), mu, sigma


In [ ]:
# Rolling Z-score baseline
z_rows = []
z_any = pd.Series(False, index=cart.index)
before_sustained = cart['timestamp'] < SUSTAINED_MEMORY_TIME

for col in FEATURES:
    mean = cart[col].rolling(window=60, min_periods=60).mean().shift(1)
    std = cart[col].rolling(window=60, min_periods=60).std().shift(1).replace(0, np.nan)
    z_col = f'{col}_rolling_z'
    flag_col = f'{col}_rolling_z_flag'
    cart[z_col] = (cart[col] - mean) / std
    cart[flag_col] = cart[z_col].abs() > 3
    z_any |= cart[flag_col].fillna(False)
    z_rows.append({
        'metric': col,
        'first_anomaly': cart.loc[cart[flag_col], 'timestamp'].min(),
        'points_flagged': int(cart[flag_col].sum()),
        'pct_flagged': round(float(cart[flag_col].mean() * 100), 2),
        'false_positive_before_16_20': int(cart.loc[before_sustained, flag_col].sum()),
    })
cart['rolling_z_any_flag'] = z_any
z_alarm = first_alarm_time(cart['rolling_z_any_flag'], cart['timestamp'])
zscore_results = pd.DataFrame(z_rows)
display(zscore_results)

# Isolation Forest secondary detector
X = StandardScaler().fit_transform(cart[FEATURES].ffill().fillna(0))
if_model = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
if_model.fit(X[:1440])
cart['if_score'] = -if_model.decision_function(X)
cart['if_flag'] = if_model.predict(X) == -1
if_alarm = first_alarm_time(cart['if_flag'], cart['timestamp'])

# CUSUM primary detector
cart['memory_cusum'], cart['memory_cusum_z'], mem_mu, mem_sigma = run_cusum(cart['memory_usage_bytes'], h=12.0, smoothing_points=120)
cart['gc_cusum'], cart['gc_cusum_z'], gc_mu, gc_sigma = run_cusum(cart['jvm_gc_pause_ms_avg'], h=10.0, smoothing_points=12)
cart['memory_cusum_flag'] = cart['memory_cusum'] > 12.0
cart['gc_cusum_flag'] = cart['gc_cusum'] > 10.0
memory_cusum_alarm = first_alarm_time(cart.loc[cart.index >= 720, 'memory_cusum_flag'], cart.loc[cart.index >= 720, 'timestamp'])
gc_cusum_alarm = first_alarm_time(cart.loc[cart.index >= 720, 'gc_cusum_flag'], cart.loc[cart.index >= 720, 'timestamp'])


In [ ]:
comparison = pd.DataFrame([
    {
        'Method': 'CUSUM (memory)',
        'Role': 'Primary continuous detector',
        'First usable alarm': memory_cusum_alarm,
        'Minutes before alert': minutes_before_alert(memory_cusum_alarm),
        'Lead time': duration_text(minutes_before_alert(memory_cusum_alarm)),
    },
    {
        'Method': 'Rolling Z-score',
        'Role': 'Explainable baseline',
        'First usable alarm': z_alarm,
        'Minutes before alert': minutes_before_alert(z_alarm),
        'Lead time': duration_text(minutes_before_alert(z_alarm)),
    },
    {
        'Method': 'Isolation Forest',
        'Role': 'Secondary multivariate detector',
        'First usable alarm': if_alarm,
        'Minutes before alert': minutes_before_alert(if_alarm),
        'Lead time': duration_text(minutes_before_alert(if_alarm)),
    },
])

ttd = pd.DataFrame([
    {'Signal': 'Earliest JVM log signal', 'Timestamp': EARLIEST_LOG_SIGNAL, 'TTD': duration_text(minutes_before_alert(EARLIEST_LOG_SIGNAL))},
    {'Signal': 'Memory trend shift', 'Timestamp': MEMORY_TREND_SHIFT, 'TTD': duration_text(minutes_before_alert(MEMORY_TREND_SHIFT))},
    {'Signal': 'Sustained memory anomaly', 'Timestamp': SUSTAINED_MEMORY_TIME, 'TTD': duration_text(minutes_before_alert(SUSTAINED_MEMORY_TIME))},
])

display(comparison)
display(ttd)
comparison.to_csv(ART / 'anomaly_method_comparison.csv', index=False)
zscore_results.to_csv(ART / 'rolling_zscore_results.csv', index=False)


In [ ]:
# Chart 1: normalized key metrics from silent signal to alert
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
for ax, col, title in zip(axes, FEATURES, ['Memory usage', 'GC pause', 'P99 latency', 'HTTP 5xx rate']):
    z = ((cart[col] - cart[col].mean()) / cart[col].std()).clip(-4, 8)
    ax.plot(cart['timestamp'], z, color='#2a6f97', linewidth=1.5)
    add_reference_lines(ax, include_trend=True)
    ax.set_ylabel('z-score')
    ax.set_title(title, loc='left')
    ax.grid(True, alpha=0.3)
axes[0].set_title('Normalized metrics - silent signal to alert', loc='center')
axes[-1].set_xlabel('Timestamp (UTC)')
fmt_time_axis(axes[-1])
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles[:3], labels[:3], loc='upper right')
fig.tight_layout(rect=[0, 0, 0.93, 1])
fig.savefig(ART / 'chart_01_correlation_normalized.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Chart 4: anomaly comparison, cleaner version
fig, axes = plt.subplots(3, 1, figsize=(16, 11), sharex=True)

axes[0].plot(cart['timestamp'], cart['memory_usage_bytes'] / (1024**3), color='#1d6996', linewidth=1.6, label='Memory usage (GB)')
axes[0].axvline(FIRST_OOMKILL, color='#c1121f', linestyle='--', linewidth=1.3, label='First OOMKilled')
axes[0].axvline(ALERT_TIME, color='black', linestyle=':', linewidth=1.3, label='Alert fired')
axes[0].set_ylabel('GB')
axes[0].set_title('Raw memory trajectory', loc='left')
axes[0].legend(loc='upper left')

axes[1].plot(cart['timestamp'], cart['memory_usage_bytes_rolling_z'], color='#ff8800', linewidth=1.4, label='Rolling Z-score')
axes[1].axhline(3, color='#d90429', linestyle='--', linewidth=1.0, label='+3 sigma')
axes[1].axhline(-3, color='#d90429', linestyle='--', linewidth=1.0, label='-3 sigma')
axes[1].scatter(cart.loc[cart['memory_usage_bytes_rolling_z_flag'], 'timestamp'], cart.loc[cart['memory_usage_bytes_rolling_z_flag'], 'memory_usage_bytes_rolling_z'], color='#d90429', s=14, zorder=3, label='Anomaly point')
axes[1].set_ylabel('Z')
axes[1].set_title('Rolling Z-score baseline detector', loc='left')
axes[1].legend(loc='upper left')

threshold = cart['if_score'].quantile(0.95)
axes[2].plot(cart['timestamp'], cart['if_score'], color='#7b2cbf', linewidth=1.4, label='IF score')
axes[2].axhline(threshold, color='#6a040f', linestyle='--', linewidth=1.0, label='95th percentile guide')
axes[2].scatter(cart.loc[cart['if_flag'], 'timestamp'], cart.loc[cart['if_flag'], 'if_score'], color='#7b2cbf', s=12, alpha=0.75, label='IF anomaly')
axes[2].axvline(ALERT_TIME, color='black', linestyle=':', linewidth=1.3, label='Alert fired')
axes[2].set_ylabel('Score')
axes[2].set_title('Isolation Forest secondary detector', loc='left')
axes[2].legend(loc='upper left')
axes[2].set_xlabel('Timestamp (UTC)')
fmt_time_axis(axes[-1])

fig.suptitle('Anomaly Detection Comparison: memory, Rolling Z-score, and Isolation Forest')
fig.tight_layout()
fig.savefig(ART / 'chart_04_anomaly_comparison.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Chart 5: cross-service error propagation
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(cart['timestamp'], cart['http_5xx_rate'], label='cart-service 5xx', color='#d90429', linewidth=1.8)
ax.plot(order['timestamp'], order['http_5xx_rate'], label='order-service 5xx', color='#f77f00', linewidth=1.5)
ax.plot(payment['timestamp'], payment['http_5xx_rate'], label='payment-service 5xx', color='#4361ee', linewidth=1.5)
ax.plot(api['timestamp'], api['cart_upstream_error_rate'], label='api-gateway cart upstream error', color='#2a9d8f', linewidth=1.6)
add_reference_lines(ax, include_trend=False)
ax.set_title('Cross-service error rate - cascade propagation')
ax.set_ylabel('Rate (%)')
ax.set_xlabel('Timestamp (UTC)')
ax.legend(loc='upper left', ncols=2)
fmt_time_axis(ax)
fig.tight_layout()
fig.savefig(ART / 'chart_05_multiservice_5xx.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Chart 6: memory vs GC scatter with trend line
x = cart['memory_usage_bytes'] / (1024**3)
y = cart['jvm_gc_pause_ms_avg']
colors = np.linspace(0, 1, len(cart))
coef = np.polyfit(x, y, 1)
poly = np.poly1d(coef)
xx = np.linspace(x.min(), x.max(), 100)

fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(x, y, c=colors, cmap='viridis', s=16, alpha=0.7)
ax.plot(xx, poly(xx), color='#c1121f', linewidth=2, label='Linear trend')
ax.axvspan(1.2, x.max() + 0.1, ymin=0.72, ymax=1.0, color='#ffb3c1', alpha=0.25)
ax.text(1.24, y.max() * 0.92, 'OOMKill zone', color='#6a040f', fontsize=11, weight='bold')
ax.set_title('Heap usage vs GC pause duration - correlation evidence')
ax.set_xlabel('Memory usage (GB)')
ax.set_ylabel('GC pause avg (ms)')
ax.legend(loc='upper left')
fig.colorbar(sc, ax=ax, label='Time progression')
fig.tight_layout()
fig.savefig(ART / 'chart_06_gc_scatter.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
# Chart 8: request rate vs latency to rule out traffic spike
fig, ax1 = plt.subplots(figsize=(16, 6))
ax2 = ax1.twinx()
ax1.plot(cart['timestamp'], cart['http_requests_per_sec'], color='#277da1', linewidth=1.6, label='Request rate (req/s)')
ax2.plot(cart['timestamp'], cart['http_p99_latency_ms'], color='#d62828', linewidth=1.6, label='P99 latency (ms)')
ax1.axvspan(MEMORY_TREND_SHIFT, ALERT_TIME, color='#f4a261', alpha=0.10)
ax1.axvline(ALERT_TIME, color='black', linestyle='--', linewidth=1.3)
ax1.set_title('Traffic vs latency - ruling out traffic spike as primary cause')
ax1.set_xlabel('Timestamp (UTC)')
ax1.set_ylabel('Requests/sec', color='#277da1')
ax2.set_ylabel('P99 latency (ms)', color='#d62828')
fmt_time_axis(ax1)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
fig.tight_layout()
fig.savefig(ART / 'chart_08_request_vs_latency.png', dpi=160, bbox_inches='tight')
plt.show()


## Ket luan chinh

- Notebook metrics nay da duoc sua lai theo bo cong nghe/method da chot.
- `CUSUM` la detector chinh trong narrative production.
- `Isolation Forest` duoc giu lam secondary multivariate detector.
- `Rolling Z-score` duoc giu vi de giai thich cho reviewer.
- Cac chart da duoc chuan hoa theo huong presentation-ready cho bai nop.
